# Dataset Splits for Training Data Preparation

[![Open In Colab](https://img.shields.io/badge/Open%20In-Colab-blue?style=for-the-badge&logo=google-colab)](https://colab.research.google.com/github/dnth/rag-datakit/blob/main/nbs/01_dataset-splits.ipynb)
[![Open In Kaggle](https://img.shields.io/badge/Open%20In-Kaggle-blue?style=for-the-badge&logo=kaggle)](https://kaggle.com/kernels/welcome?src=https://github.com/dnth/rag-datakit/blob/main/nbs/01_dataset-splits.ipynb)

This notebook prepares synthetic training data for embedding model training by creating proper train/validation splits. We'll work with the synthetic triplets generated from the Singapore SkillsFuture Framework dataset to create training and validation sets suitable for embedding model fine-tuning.

## What you'll learn:
- How to load synthetic datasets from Hugging Face Hub
- Creating proper train/validation splits for embedding training
- Data preprocessing for triplet-based training
- Publishing split datasets back to Hugging Face Hub

## Installation

Install the rag-datakit package which includes all necessary dependencies including distilabel, transformers, and dataset utilities. Uncomment the cell below to install if you haven't already.

In [ ]:
# !pip install git+https://github.com/dnth/rag-datakit.git

## Dataset Loading and Preprocessing

Load the synthetic dataset generated from the previous notebook. We'll work with the triplet data (anchor, positive, negative) that was created using distilabel for embedding training purposes.

**Dataset source**: Synthetic job description triplets from Singapore Skills Framework  
**HuggingFace repo**: `dnth/ssf-synthetic-data-for-retriever-openai`

The dataset contains triplets for embedding training:
- **anchor**: Original job role descriptions
- **positive**: Paraphrased versions (semantically similar)
- **negative**: Different job descriptions (semantically dissimilar)

In [3]:
from datasets import load_dataset, concatenate_datasets

dataset_easy = load_dataset("dnth/ssf-dataset-synthetic-v2", "easy_triplets")
dataset_hard = load_dataset("dnth/ssf-dataset-synthetic-v2", "hard_triplets")


hard_triplets/train-00000-of-00001.parqu(…):   0%|          | 0.00/4.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1885 [00:00<?, ? examples/s]

In [4]:
dataset = concatenate_datasets([dataset_easy['train'], dataset_hard['train']])
dataset

Dataset({
    features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
    num_rows: 3770
})

In [5]:
dataset = dataset.select_columns(['anchor', 'positive', 'negative'])
dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 3770
})

## Data Preparation and Shuffling

Prepare the dataset for training by:
1. Selecting only the essential columns needed for embedding training (anchor, positive, negative)
2. Shuffling the data to ensure random distribution for better training

This step removes metadata columns that aren't needed for model training, keeping only the triplet data that will be used to train the embedding model.

In [6]:
dataset = dataset.shuffle()
dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 3770
})

## Train/Validation Split Creation

Create an 80/20 split of the dataset for training and validation:
- **Training set (80%)**: Used to train the embedding model
- **Validation set (20%)**: Used to evaluate model performance during training

This split ensures we have sufficient data for training while maintaining a holdout set for validation. The validation set helps monitor training progress and prevents overfitting.

In [7]:
train_size = int(0.8 * len(dataset))
valid_size = len(dataset) - train_size

train_dataset = dataset.select(range(train_size))
valid_dataset = dataset.select(range(train_size, train_size + valid_size))

In [8]:
train_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 3016
})

In [9]:
valid_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 754
})

## Publishing Final Dataset

Upload the properly split dataset to Hugging Face Hub for easy access during embedding model training. The dataset will be structured as a `DatasetDict` with separate train and validation splits.

This makes the dataset readily available for:
- Embedding model fine-tuning scripts
- Reproducible training experiments  
- Sharing with team members or the community
- Integration with training frameworks like sentence-transformers

In [10]:
from datasets import DatasetDict

ds = DatasetDict({
    "train": train_dataset,
    "valid": valid_dataset
})

ds

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 3016
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 754
    })
})

In [11]:
ds.push_to_hub("dnth/ssf-train-valid-v2")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  16%|#5        |  526kB / 3.39MB            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  62%|######1   |  525kB /  849kB            

CommitInfo(commit_url='https://huggingface.co/datasets/dnth/ssf-train-valid-v2/commit/fb19ed3200e91badb0cb224e0debe909f1d7c2bd', commit_message='Upload dataset', commit_description='', oid='fb19ed3200e91badb0cb224e0debe909f1d7c2bd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/dnth/ssf-train-valid-v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='dnth/ssf-train-valid-v2'), pr_revision=None, pr_num=None)